# **🎧Music Taste Analysis🔊**

### **Exploratory Data Analysis**

#### **Dataset:** Spotify saved tracks.
#### **Source:** Spotify Web API.
#### **Collection Date:** 23rd October 2026.
#### **Observations:** 14,175 tracks.

## **1. Introduction**
#### This notebook explores the composition and development of my personal Spotify music library. The aim is to identify patterns in artist preferences, release periods, listening interests and the evolution of my saved music collection.

#### The analysis will initially focus on descriptive statistics and visualisations, providing a foundation for more advanced investigations into musical preferences, genre clustering and recommendations.

### **Dataset Overview**
#### The dataset was collected using a custom Python script (`src/data_collection.py`) and contains track-level metadata, including artist names, album names, release dates, durations and the dates which each track was saved. 

#### **Important Limitations**
#### - Saved tracks represent library membership, not listening frequency or play counts.
#### - Spotify's popularity metric is unavailable as of 06/02/2026.
#### - Audio features such as tempo, energy, and danceability are currently not included.
#### - The dataset represents a snapshot of the library at the time of collection.

In [2]:
# Import dependencies

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import plotly.express as px

In [3]:
df = pd.read_csv(
    "../data/saved_tracks.csv"
)
print("Loading Complete")
df.head()

Loading Complete


,track_id,track_name,artist_names,album_name,release_date,popularity,duration_ms,explicit,added_at
0,3pKvmMPEFAtqpjGFyOrJy2,Ad Misericordiam,µ-Ziq,Tango N'Vectif,2006-07-03,NaN,209426,False,2026-10-23T13:01:15Z
1,3BSsT1tMUMyt72NdKUL8bL,The Prizefighter,Slo Burn,Amusing The Amazing,1997-04-15,NaN,136160,False,2026-10-23T10:21:45Z
2,19QzzIsx0tPAvMQwiQmESl,Tennessee Porch Swing,Sir Richard Bishop,Polytheistic Fragments,2007-09-25,NaN,117066,False,2026-10-22T16:02:06Z
3,6ZUsVj3kuS0LY2izJJcymE,Apothecaries' Weight,Kyuss,Blues for the Red Sun,1992-01-01,NaN,321266,False,2026-10-22T15:37:53Z
4,4qCjs9O9rjhe8YIQgiIswu,"Izabella - Live at the Fillmore East, NY - 12/...",Jimi Hendrix,Songs For Groovy Children: The Fillmore East C...,2019-11-22,NaN,208521,False,2026-10-21T21:56:47Z


## **2. Data Quality & Overview**
#### Before exploring musical preferences, the dataset is assessed for completeness, uniqueness and consistency. This'll identify any limitations requiring attention.

In [4]:
print(df.shape, )
print(df.isna().sum())
print(f"there are {df.duplicated().sum()} missing observations")

(14175, 9)
track_id            0
track_name          0
artist_names        0
album_name          0
release_date        0
popularity      14175
duration_ms         0
explicit            0
added_at            0
dtype: int64
there are 0 missing observations


In [5]:
df.dtypes

track_id            str
track_name          str
artist_names        str
album_name          str
release_date        str
popularity      float64
duration_ms       int64
explicit           bool
added_at            str
dtype: object

#### Will be adding a new column for song length to be in mins & seconds, will *kill* the popularity column, since it's redundant, and finally will convert the date added to datetime and extract year from release date.

In [6]:
df["added_at"] = pd.to_datetime(df["added_at"], utc = True)
print(df["added_at"].dtype)

datetime64[us, UTC]


In [7]:
df["duration_min"] = (df["duration_ms"] / 60000)
df["release_year"] = df["release_date"].str[:4].astype(int)
df = df.drop(columns = ["popularity"])

## **3. Library Composition**
### **3.1 Most Represented Artists**

In [8]:
df["artist_names"].value_counts().head(10)

artist_names
Neil Young                      164
Neil Young, Crazy Horse         122
The Clash                       104
Dom & Roland                     98
The Brian Jonestown Massacre     95
Pink Floyd                       94
The Fall                         94
The Rolling Stones               81
Grandaddy                        60
Fleetwood Mac                    60
Name: count, dtype: int64

#### Now, early on we've run into an issue. Mr. Neil is represented twice. I think the fairest way to proceed is to credit each named artist seperately, and then count artist appearances. 

In [9]:
df["artists_split"] = df["artist_names"].str.split(",")
artists_exploded = df["artists_split"].explode()
artists_exploded = artists_exploded.str.strip()
artists_exploded.value_counts().head(10)

artists_split
Neil Young                      304
Technical Itch                  135
Dom & Roland                    131
Crazy Horse                     127
The Clash                       105
The Brian Jonestown Massacre     96
Pink Floyd                       95
The Fall                         94
The Rolling Stones               83
Brakken                          63
Name: count, dtype: int64

In [12]:
df["primary_artist"] = df["artists_split"].str[0]
df[["artist_names", "primary_artist"]].head(10)

,artist_names,primary_artist
0,µ-Ziq,µ-Ziq
1,Slo Burn,Slo Burn
2,Sir Richard Bishop,Sir Richard Bishop
3,Kyuss,Kyuss
4,Jimi Hendrix,Jimi Hendrix
5,Ten Years After,Ten Years After
6,Ten Years After,Ten Years After
7,Built To Spill,Built To Spill
8,Raphael Roginski,Raphael Roginski
9,本田竹曠,本田竹曠
